# 03 - Chain Complexes and Homology from Scratch

In this notebook, we move from combinatorial structure to exact linear algebra. A `ChainComplex` represents the topology of a space purely as a sequence of integer boundary matrices. We will learn how to extract these matrices, compute exact homology groups (including torsion), and visualize the resulting homology generators on small figures.

## Learning Goals
- **Boundary Matrices**: Understand how a `SimplicialComplex` is converted into sparse integer matrices.
- **Exact Homology**: Compute Betti numbers and torsion coefficients over $\mathbb{Z}$.
- **Homology Generators**: Extract specific cycles that generate the homology groups $H_n(X)$.
- **Visualization**: Use Plotly to visualize these generating cycles directly on the 3D geometry of the complex.

## Formal Grounding

### The Chain Complex
A chain complex $(C_*, \partial_*)$ is a sequence of abelian groups and boundary maps:
$$ \dots \to C_{n+1} \xrightarrow{\partial_{n+1}} C_n \xrightarrow{\partial_n} C_{n-1} \to \dots $$
where $\partial_n \circ \partial_{n+1} = 0$.

### Homology and Generators
The $n$-th homology group is defined as the quotient of cycles by boundaries:
$$H_n = \ker(\partial_n)/\operatorname{im}(\partial_{n+1}).$$
A **Homology Generator** is a specific cycle $z \in \ker(\partial_n)$ that represents a non-trivial equivalence class in $H_n$.


In [1]:
import numpy as np
import plotly.graph_objects as go
import pysurgery as ps

from pysurgery.core.homology_generators import (
    compute_homology_basis_from_complex,
    compute_optimal_h1_basis_from_complex
)
from pysurgery.core.fundamental_group import extract_pi_1_with_traces

from pysurgery.bridge.julia_bridge import julia_engine

# Performance Check: Julia Acceleration
if julia_engine.available:
    print('Warming up Julia engine...')
    julia_engine.warmup()
else:
    print('Julia backend not available, falling back to NumPy.')

print('=' * 70)
print('03 - Chain Complexes and Homology: Setup Complete')
print('=' * 70)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Julia backend not available, falling back to NumPy.
03 - Chain Complexes and Homology: Setup Complete


## Part 1: From Simplices to Boundary Matrices

Let's build a simple complex (a filled triangle) and examine its boundary matrices.


### Example 3.1: The Triangle Boundary

In [2]:
# A single 2-simplex (triangle)
sc_tri = ps.SimplicialComplex.from_maximal_simplices([(0, 1, 2)])
cc = sc_tri.cellular_chain_complex()

print("Chain Complex dimensions:", cc.dimensions)
for n in [1, 2]:
    mat = cc.boundaries[n].toarray()
    print(f"\nBoundary d_{n} matrix (Shape {mat.shape}):")
    print(mat)
    print(f"Rank of d_{n}: {np.linalg.matrix_rank(mat)}")

Chain Complex dimensions: [0, 1, 2]

Boundary d_1 matrix (Shape (3, 3)):
[[-1 -1  0]
 [ 1  0 -1]
 [ 0  1  1]]
Rank of d_1: 2

Boundary d_2 matrix (Shape (3, 1)):
[[ 1]
 [-1]
 [ 1]]
Rank of d_2: 1


## Part 2: Exact Homology and Torsion

Homology measures the "holes" in a space. We compute it exactly over $\mathbb{Z}$ to capture both free rank (Betti numbers) and torsion.


### Example 3.2: Homology of canonical spaces

In [3]:
# Build standard spaces
spaces = {
    'Triangle': ps.SimplicialComplex.from_maximal_simplices([(0, 1, 2)]),
    'Circle': ps.SimplicialComplex.from_maximal_simplices([(0,1), (1,2), (2,0)]),
    'Torus': ps.SimplicialComplex.from_maximal_simplices([
        (0, 3, 4), (0, 1, 4),
        (1, 4, 5), (1, 2, 5),
        (2, 3, 5), (0, 2, 3),
        (3, 6, 7), (3, 4, 7),
        (4, 7, 8), (4, 5, 8),
        (5, 6, 8), (3, 5, 6),
        (0, 1, 6), (1, 6, 7),
        (1, 2, 7), (2, 7, 8),
        (0, 2, 8), (0, 6, 8),
    ])
}

for name, sc in spaces.items():
    cc = sc.cellular_chain_complex()
    print(f'\n{name} Homology:')
    for n in range(3):
        rank, torsion = cc.homology(n)
        torsion_str = f" + Z_{torsion[0]}" if torsion else ""
        print(f"  H_{n} = Z^{rank}{torsion_str}")

/home/gabriel/Desktop/SurgeryTheory/pysurgery/core/complexes.py:639: UserWarning: Torsion certification may be incomplete for this complex; the sparse integer reduction returned only unit factors, so torsion could not be fully resolved.
  warnings.warn(



Triangle Homology:
  H_0 = Z^1
  H_1 = Z^0
  H_2 = Z^0

Circle Homology:
  H_0 = Z^1
  H_1 = Z^1
  H_2 = Z^0

Torus Homology:
  H_0 = Z^1
  H_1 = Z^2
  H_2 = Z^1


## Part 3: Homology Generators on Small Figures

Finding the rank of $H_n$ tells us *how many* holes there are. **Generators** tell us *where* they are. We use GUDHI to build the simplex tree and extract visualizable generators for $H_0$, $H_1$, and $H_2$.


### Example 3.3: Visualizing Generators (Torus)

In [6]:
import gudhi

# Build a small Torus point cloud and complex
R, r = 2.0, 1.0
nu, nv = 20, 20
pts = []
simplices = []

def idx(i, j): return i * nv + j

for i in range(nu):
    u = 2.0 * np.pi * i / nu
    for j in range(nv):
        v = 2.0 * np.pi * j / nv
        pts.append([(R + r * np.cos(v)) * np.cos(u), (R + r * np.cos(v)) * np.sin(u), r * np.sin(v)])

for i in range(nu):
    ip = (i + 1) % nu
    for j in range(nv):
        jp = (j + 1) % nv
        a, b, c, d = idx(i, j), idx(ip, j), idx(ip, jp), idx(i, jp)
        simplices.extend([(a, b, c), (a, c, d)])

st = gudhi.SimplexTree()
for v in range(len(pts)):
    st.insert([v])
for s in simplices:
    st.insert(list(s))
pts = np.array(pts)
st = ps.SimplicialComplex.from_gudhi_simplex_tree(st)

# Extract Generators
h0 = compute_homology_basis_from_complex(st, dimension=0)
h1 = compute_optimal_h1_basis_from_complex(st, point_cloud=pts)
h2 = compute_homology_basis_from_complex(st, dimension=2, mode="valid")

print(f"H0 Generators Found: {h0.rank}")
print(f"H1 Generators Found: {h1.rank}")
print(f"H2 Generators Found: {h2.rank}")

# Plotting with Plotly
fig = go.Figure()
faces = [s for s in st.n_simplices(2) if len(s) == 3]

# Base semi-transparent surface
fig.add_trace(go.Mesh3d(
    x=pts[:,0], y=pts[:,1], z=pts[:,2],
    i=[f[0] for f in faces], j=[f[1] for f in faces], k=[f[2] for f in faces],
    opacity=0.1, color="lightblue", name="Base Surface"
))

# Plot H0 (vertices)
h0_vs = []
for gen in h0.generators:
    for simplex in gen.support_simplices:
        if len(simplex) == 1:
            h0_vs.append(int(simplex[0]))
h0_vs = list(set(h0_vs))
if h0_vs:
    fig.add_trace(go.Scatter3d(
        x=pts[h0_vs, 0], y=pts[h0_vs, 1], z=pts[h0_vs, 2],
        mode="markers", marker=dict(size=10, color="black", symbol="diamond"),
        name="H0 generator(s)"
    ))

# Plot H1 (cycles)
colors = ["red", "purple"]
for gi, gen in enumerate(h1.generators[:2]):
    edges = [tuple(sorted(e)) for e in gen.support_edges]
    xs, ys, zs = [], [], []
    for u, v in edges:
        xs.extend([pts[u,0], pts[v,0], None])
        ys.extend([pts[u,1], pts[v,1], None])
        zs.extend([pts[u,2], pts[v,2], None])
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs, mode="lines",
        line=dict(width=6, color=colors[gi]),
        name=f"H1 Generator {gi}"
    ))

# Plot H2 (cavities/faces)
h2_faces = []
for gen in h2.generators:
    for simplex in gen.support_simplices:
        if len(simplex) == 3:
            h2_faces.append(tuple(sorted(simplex)))
h2_faces = list(set(h2_faces))
if h2_faces:
    fig.add_trace(go.Mesh3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        i=[f[0] for f in h2_faces], j=[f[1] for f in h2_faces], k=[f[2] for f in h2_faces],
        opacity=0.6, color="gold", name="H2 generator support"
    ))

fig.update_layout(title="Torus H0, H1, and H2 Generators", height=600)
fig.show()

H0 Generators Found: 1
H1 Generators Found: 2
H2 Generators Found: 1


### Quick Mapper Application

Recently introduced, the `quick_mapper` method for simplex trees preserves topology while reducing complexity. Here we build a Torus, apply `quick_mapper`, use `expand` to increase its simplex tree to 2 dimensions, compute the homology generators of all degrees and the fundamental group, and plot its vertices, edges, and generators using plotly.

In [7]:
# 2. Apply quick_mapper and expand to 2 dimensions
sc_qm, _ = st.quick_mapper(preserve_topology=True, max_loops=3)
sc_qm = sc_qm.expand(2)

# 3. Compute Homology Generators
h0_qm = compute_homology_basis_from_complex(sc_qm, dimension=0)
#h1_qm = compute_optimal_h1_basis_from_complex(st_qm, point_cloud=pts_qm)
h1_qm = compute_homology_basis_from_complex(sc_qm, dimension=1, mode="optimal")
h2_qm = compute_homology_basis_from_complex(sc_qm, dimension=2, mode="optimal")

print(f"Quick Mapper H0 Generators: {h0_qm.rank}")
print(f"Quick Mapper H1 Generators: {h1_qm.rank}")
print(f"Quick Mapper H2 Generators: {h2_qm.rank}")

# 4. Compute Fundamental Group Generators
boundaries = sc_qm.chain_complex().boundaries
cw = ps.CWComplex(cells=sc_qm.cells_count, attaching_maps=boundaries, coefficient_ring="Z")
pi1_with_traces = extract_pi_1_with_traces(cw, simplify=True, generator_mode="optimized")

# 5. Plotting
fig_qm = go.Figure()

# Plot Quick Mapper Vertices and Edges
qm_vertices = list(sc_qm.n_simplices(0))
qm_v_coords = np.array([pts_qm[v[0]] for v in qm_vertices])

fig_qm.add_trace(go.Scatter3d(
    x=qm_v_coords[:, 0], y=qm_v_coords[:, 1], z=qm_v_coords[:, 2],
    mode="markers", marker=dict(size=8, color="black"),
    name="Quick Mapper Vertices"
))

qm_edges = list(sc_qm.n_simplices(1))
xs, ys, zs = [], [], []
for u, v in qm_edges:
    xs.extend([pts_qm[u, 0], pts_qm[v, 0], None])
    ys.extend([pts_qm[u, 1], pts_qm[v, 1], None])
    zs.extend([pts_qm[u, 2], pts_qm[v, 2], None])

fig_qm.add_trace(go.Scatter3d(
    x=xs, y=ys, z=zs, mode="lines",
    line=dict(width=3, color="lightgray"), name="Quick Mapper Edges"
))

# Plot H1 cycles
colors_h1 = ["red", "purple", "green", "orange"]
for gi, gen in enumerate(h1_qm.generators[:2]):
    edges = [tuple(sorted(e)) for e in gen.support_edges]
    hx, hy, hz = [], [], []
    for u, v in edges:
        hx.extend([pts_qm[u,0], pts_qm[v,0], None])
        hy.extend([pts_qm[u,1], pts_qm[v,1], None])
        hz.extend([pts_qm[u,2], pts_qm[v,2], None])
    fig_qm.add_trace(go.Scatter3d(
        x=hx, y=hy, z=hz, mode="lines",
        line=dict(width=7, color=colors_h1[gi % len(colors_h1)]),
        name=f"H1 Generator {gi}"
    ))

# Plot pi_1 traces
for i, tr in enumerate(pi1_with_traces.traces):
    path = tr.undirected_edge_path
    tx, ty, tz = [], [], []
    for u, v in path:
        tx.extend([pts_qm[int(u), 0], pts_qm[int(v), 0], None])
        ty.extend([pts_qm[int(u), 1], pts_qm[int(v), 1], None])
        tz.extend([pts_qm[int(u), 2], pts_qm[int(v), 2], None])
    fig_qm.add_trace(go.Scatter3d(
        x=tx, y=ty, z=tz, mode="lines",
        line=dict(width=5, dash='dash', color=colors_h1[(i+2) % len(colors_h1)]),
        name=f"pi_1 Generator {tr.generator}"
    ))

# Plot H2 faces (semi-transparent)
h2_faces_qm = []
for gen in h2_qm.generators:
    for simplex in gen.support_simplices:
        if len(simplex) == 3:
            h2_faces_qm.append(tuple(sorted(simplex)))
h2_faces_qm = list(set(h2_faces_qm))

if h2_faces_qm:
    fig_qm.add_trace(go.Mesh3d(
        x=pts_qm[:, 0], y=pts_qm[:, 1], z=pts_qm[:, 2],
        i=[f[0] for f in h2_faces_qm], j=[f[1] for f in h2_faces_qm], k=[f[2] for f in h2_faces_qm],
        opacity=0.5, color="gold", name="H2 generator support"
    ))

fig_qm.update_layout(title="Quick Mapper Homology and Fundamental Group on Torus")
fig_qm.show()

ValueError: Topological invalidity: Edge (1, 356) contains a vertex outside the bounds [0, 74].

## Failure Modes

1. **Non-Orientable Torsion**: Some algorithms compute homology over $\mathbb{Q}$ or $\mathbb{Z}_2$, which loses integer torsion information (e.g., the $\mathbb{Z}_2$ in $H_1(\mathbb{RP}^2)$). `pySurgery` defaults to exact $\mathbb{Z}$ algebra.
2. **Optimal Cycle Complexity**: Finding the strictly *shortest* homology generator is NP-hard. The `compute_optimal_h1_basis` uses greedy approximations over the 1-skeleton graph.


In [ ]:
try:
    # Attempting to extract homology without Julia may fallback to slower SymPy paths
    pass
except Exception as e:
    print(f"Error: {e}")

## Summary Checklist
- [x] Extracted exact integer boundary matrices from a `SimplicialComplex`.
- [x] Computed exact Betti numbers and torsion for basic manifolds.
- [x] Generated and visualized optimal $H_0$, $H_1$, and $H_2$ cycles using GUDHI and Plotly.

## Exercises
1. **The Klein Bottle**: Construct a Klein bottle and compute its exact integer homology. Can you spot the torsion?
2. **Two Loops**: Create a space with two disjoint circle loops. Visualize its $H_1$ generators.
3. **Sphere H2**: Use `compute_homology_basis_from_simplex_tree` with `dimension=2` on a triangulated sphere and plot the resulting 2-simplices.

**Ready for [04 - Cohomology, Cup Products, and Intuition](./04_cohomology_cup_products_and_intuition.ipynb)**
